# 🔄 ETL Silver → Gold (Spark Job)
## Crime Data Pipeline

Job de transformação para execução via Airflow/Spark.

**Objetivo**: Criar modelo dimensional (Star Schema) na camada Gold para analytics e BI.

In [ ]:
# Configurações do Spark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

# Iniciar sessão Spark
spark = SparkSession.builder \
    .appName("SilverToGold_CrimeData") \
    .getOrCreate()

print(f"✅ Spark Session iniciada: {spark.version}")

In [ ]:
# Configuração de caminhos (detectar raiz do projeto)
import os
from pathlib import Path

def find_project_root() -> Path:
    """Encontra a raiz do projeto SBD2"""
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    for root in candidates:
        if (root / 'Crime_Data_from_2020_to_Present.csv').exists():
            return root
        if (root / 'data').exists() and (root / 'notebooks').exists():
            return root
    return cwd

PROJECT_ROOT = find_project_root()
SILVER_PATH = str(PROJECT_ROOT / "data" / "silver")
GOLD_PATH = str(PROJECT_ROOT / "data" / "gold")
BATCH_ID = datetime.now().strftime('%Y%m%d_%H%M%S')

# Garantir que diretório existe
os.makedirs(GOLD_PATH, exist_ok=True)

print(f"📁 Projeto: {PROJECT_ROOT}")
print(f"📁 Silver: {SILVER_PATH}")
print(f"📁 Gold: {GOLD_PATH}")
print(f"🔖 Batch: {BATCH_ID}")

In [ ]:
# Carregar dados Silver
df_silver = spark.read.parquet(f"{SILVER_PATH}/crimes.parquet")
print(f"✅ Dados carregados: {df_silver.count():,} registros")

In [ ]:
# Criar Dimensão: Data
dim_date = df_silver \
    .select(F.to_date("date_occurred").alias("full_date")) \
    .distinct() \
    .withColumn("sk_date", F.monotonically_increasing_id() + 1) \
    .withColumn("year", F.year("full_date")) \
    .withColumn("quarter", F.quarter("full_date")) \
    .withColumn("month", F.month("full_date")) \
    .withColumn("month_name", F.date_format("full_date", "MMMM")) \
    .withColumn("week_of_year", F.weekofyear("full_date")) \
    .withColumn("day_of_month", F.dayofmonth("full_date")) \
    .withColumn("day_of_week", F.dayofweek("full_date")) \
    .withColumn("day_name", F.date_format("full_date", "EEEE")) \
    .withColumn("is_weekend", F.when(F.dayofweek("full_date").isin([1, 7]), True).otherwise(False))

dim_date.write.mode("overwrite").parquet(f"{GOLD_PATH}/dim_date.parquet")
print(f"✅ dim_date: {dim_date.count():,} registros")

In [ ]:
# Criar Dimensão: Tempo
dim_time = spark.range(24) \
    .withColumnRenamed("id", "hour") \
    .withColumn("sk_time", F.col("hour") + 1) \
    .withColumn("period_of_day",
                F.when(F.col("hour") < 6, "MADRUGADA")
                 .when(F.col("hour") < 12, "MANHA")
                 .when(F.col("hour") < 18, "TARDE")
                 .otherwise("NOITE")) \
    .withColumn("is_rush_hour", F.col("hour").isin([7, 8, 9, 17, 18, 19]))

dim_time.write.mode("overwrite").parquet(f"{GOLD_PATH}/dim_time.parquet")
print(f"✅ dim_time: {dim_time.count():,} registros")

In [ ]:
# Criar Dimensão: Área
dim_area = df_silver \
    .select("area_code", "area_name") \
    .distinct() \
    .withColumn("sk_area", F.monotonically_increasing_id() + 1) \
    .withColumn("region",
                F.when(F.col("area_name").isin(["DEVONSHIRE", "FOOTHILL", "MISSION", "NORTH HOLLYWOOD", "VAN NUYS", "WEST VALLEY"]), "NORTH")
                 .when(F.col("area_name").isin(["77TH STREET", "HARBOR", "SOUTHEAST", "SOUTHWEST"]), "SOUTH")
                 .when(F.col("area_name").isin(["CENTRAL", "HOLLENBECK", "RAMPART"]), "CENTRAL")
                 .when(F.col("area_name").isin(["HOLLYWOOD", "OLYMPIC", "PACIFIC", "WEST LA", "WILSHIRE"]), "WEST")
                 .otherwise("OTHER"))

dim_area.write.mode("overwrite").parquet(f"{GOLD_PATH}/dim_area.parquet")
print(f"✅ dim_area: {dim_area.count():,} registros")

In [ ]:
# Criar Dimensão: Tipo de Crime
dim_crime_type = df_silver \
    .select("crime_code", "crime_description", "is_violent") \
    .distinct() \
    .withColumn("sk_crime_type", F.monotonically_increasing_id() + 1) \
    .withColumn("crime_category",
                F.when(F.col("crime_description").contains("THEFT"), "THEFT")
                 .when(F.col("crime_description").contains("ASSAULT"), "ASSAULT")
                 .when(F.col("crime_description").contains("VEHICLE"), "VEHICLE")
                 .when(F.col("crime_description").contains("ROBBERY"), "ROBBERY")
                 .otherwise("OTHER")) \
    .withColumn("severity_level", F.when(F.col("is_violent") == 1, 3).otherwise(1))

dim_crime_type.write.mode("overwrite").parquet(f"{GOLD_PATH}/dim_crime_type.parquet")
print(f"✅ dim_crime_type: {dim_crime_type.count():,} registros")

In [ ]:
# Criar Dimensão: Vítima
dim_victim = df_silver \
    .select("victim_age", "victim_sex", "victim_descent") \
    .distinct() \
    .withColumn("sk_victim", F.monotonically_increasing_id() + 1) \
    .withColumn("age_group",
                F.when(F.col("victim_age") < 18, "MENOR")
                 .when(F.col("victim_age") < 30, "JOVEM")
                 .when(F.col("victim_age") < 45, "ADULTO")
                 .when(F.col("victim_age") < 60, "MEIA_IDADE")
                 .otherwise("IDOSO"))

dim_victim.write.mode("overwrite").parquet(f"{GOLD_PATH}/dim_victim.parquet")
print(f"✅ dim_victim: {dim_victim.count():,} registros")

In [ ]:
# Criar Tabela Fato: Crimes
# Primeiro, criar os mapeamentos
date_map = dim_date.select("full_date", "sk_date")
time_map = dim_time.select("hour", "sk_time")
area_map = dim_area.select("area_code", "sk_area")
crime_type_map = dim_crime_type.select("crime_code", "sk_crime_type")
victim_map = dim_victim.select("victim_age", "victim_sex", "victim_descent", "sk_victim")

# Criar fato com joins
fato_crimes = df_silver \
    .withColumn("full_date", F.to_date("date_occurred")) \
    .join(date_map, "full_date", "left") \
    .join(time_map, df_silver["hour_occurred"] == time_map["hour"], "left") \
    .join(area_map, "area_code", "left") \
    .join(crime_type_map, "crime_code", "left") \
    .join(victim_map, ["victim_age", "victim_sex", "victim_descent"], "left") \
    .select(
        F.monotonically_increasing_id().alias("sk_crime"),
        F.col("crime_id").alias("nk_crime_id"),
        "sk_date",
        "sk_time",
        "sk_area",
        "sk_crime_type",
        "sk_victim",
        "latitude",
        "longitude",
        "is_violent"
    )

fato_crimes.write.mode("overwrite").parquet(f"{GOLD_PATH}/fato_crimes.parquet")
print(f"✅ fato_crimes: {fato_crimes.count():,} registros")

In [ ]:
# Criar Agregação: Crimes por Área e Ano/Mês
agg_crimes_area_year = fato_crimes \
    .join(dim_date.select("sk_date", "year", "month"), "sk_date") \
    .groupBy("sk_area", "year", "month") \
    .agg(
        F.count("*").alias("total_crimes"),
        F.sum("is_violent").alias("violent_crimes")
    )

agg_crimes_area_year.write.mode("overwrite").parquet(f"{GOLD_PATH}/agg_area_month.parquet")
print(f"✅ agg_area_month: {agg_crimes_area_year.count():,} registros")

In [ ]:
# Criar Agregação: Crimes por tipo e ano
agg_type_year = fato_crimes \
    .join(dim_date.select("sk_date", "year"), "sk_date") \
    .join(dim_crime_type.select("sk_crime_type", "crime_description"), "sk_crime_type") \
    .groupBy("crime_description", "year") \
    .agg(
        F.count("*").alias("total_crimes"),
        F.sum("is_violent").alias("violent_crimes")
    )

agg_type_year.write.mode("overwrite").parquet(f"{GOLD_PATH}/agg_type_year.parquet")
print(f"✅ agg_type_year: {agg_type_year.count():,} registros")

In [ ]:
# Finalizar
spark.stop()
print("\n" + "="*50)
print("✅ Job Silver → Gold concluído com sucesso!")
print("="*50)